In [2]:
// :clear

In [3]:
:env LLVM_SYS_140_PREFIX=/usr/lib/llvm-14
:restart
:dep pecos = { version = "0.1.1", path = "../crates/pecos" }

Set LLVM_SYS_140_PREFIX=/usr/lib/llvm-14 (use :restart command to reload child process)
Child process restarted


In [4]:
// Bell State Example using SparseStab simulator
use pecos::prelude::*;

// Create a 2-qubit stabilizer simulator
let mut sim = StdSparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sim.h(0)      // Apply Hadamard to qubit 0: |00⟩ -> (|00⟩ + |10⟩)/√2
   .cx(0, 1); // Apply CNOT: (|00⟩ + |10⟩)/√2 -> (|00⟩ + |11⟩)/√2

// Measure both qubits
let result0: MeasurementResult = sim.mz(0);
let result1: MeasurementResult = sim.mz(1);

println!("Bell State Measurement Results:");
println!("  Qubit 0: {} (deterministic: {})", 
         if result0.outcome { "1" } else { "0" },
         result0.is_deterministic);
println!("  Qubit 1: {} (deterministic: {})", 
         if result1.outcome { "1" } else { "0" },
         result1.is_deterministic);

// Verify Bell state correlation: both qubits should always measure the same
assert_eq!(result0.outcome, result1.outcome, 
           "Bell state measurements must be correlated!");
println!("\nSuccess! Measurements are correlated as expected for a Bell state.");

Bell State Measurement Results:
  Qubit 0: 1 (deterministic: false)
  Qubit 1: 1 (deterministic: true)



In [5]:
// Symbolic Bell State Example using SymbolicSparseStab
// This simulator tracks measurement dependencies instead of collapsing to concrete outcomes

let mut sym_sim = StdSymbolicSparseStab::new(2);

// Create Bell state: (|00⟩ + |11⟩)/√2
sym_sim.h(0).cx(0, 1);

// Measure both qubits
let r0: SymbolicMeasurementResult = sym_sim.mz(0);
let r1: SymbolicMeasurementResult = sym_sim.mz(1);

println!("Symbolic Bell State Measurement Results:");
println!("  Measurement 0: {:?} (deterministic: {})", r0.outcome, r0.is_deterministic);
println!("  Measurement 1: {:?} (deterministic: {})", r1.outcome, r1.is_deterministic);

// The first measurement is non-deterministic (creates measurement index 0)
// The second measurement is deterministic and depends on measurement 0
println!("\nAnalysis:");
if !r0.is_deterministic {
    println!("  - Qubit 0 measurement was non-deterministic (random outcome)");
}
if r1.is_deterministic && r0.outcome == r1.outcome {
    println!("  - Qubit 1 measurement is deterministic and depends on measurement 0");
    println!("  - This shows the Bell state correlation: both qubits always measure the same!");
}

Success! Measurements are correlated as expected for a Bell state.
Symbolic Bell State Measurement Results:
  Measurement 0: {0} (deterministic: false)
  Measurement 1: {0} (deterministic: true)

Analysis:
  - Qubit 0 measurement was non-deterministic (random outcome)
  - Qubit 1 measurement is deterministic and depends on measurement 0
  - This shows the Bell state correlation: both qubits always measure the same!


()

In [15]:
// 3-qubit Repetition Code in Logical |+_L⟩ with Syndrome Measurements
//
// Qubits:
//   q0, q1, q2: Data qubits 
//   q3: Ancilla for Z0Z1 check
//   q4: Ancilla for Z1Z2 check
//
// Logical states:
//   |0_L⟩ = |000⟩
//   |1_L⟩ = |111⟩
//   |+_L⟩ = (|000⟩ + |111⟩)/√2
//
// Encoding circuit for |+_L⟩:
//   - Start with |+⟩|00⟩ (H on q0)
//   - CX(0,1), CX(0,2) to spread → (|000⟩ + |111⟩)/√2
//
// Syndrome extraction:
//   - Measure Z0Z1 using ancilla q3
//   - Measure Z1Z2 using ancilla q4
//
// Then measure data qubits in Z basis

let mut sim = StdSymbolicSparseStab::new(5);

println!("=== 3-Qubit Repetition Code: Logical |+_L⟩ ===\n");

// Encode logical |+_L⟩ = (|000⟩ + |111⟩)/√2
sim.h(0);           // |+⟩|00⟩
sim.cx(0, 1);       // (|00⟩ + |11⟩)|0⟩
sim.cx(0, 2);       // |000⟩ + |111⟩
println!("After encoding |+_L⟩ = (|000⟩ + |111⟩)/√2:");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z0Z1 via ancilla q3
// Circuit: H(q3), CX(q0,q3), CX(q1,q3), H(q3), Measure(q3)
sim.h(3);
sim.cx(0, 3);
sim.cx(1, 3);
sim.h(3);
println!("After Z0Z1 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s0: SymbolicMeasurementResult = sim.mz(3);
println!("Syndrome S0 (Z0Z1) = {:?}, det={}", s0.outcome, s0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Syndrome measurement: Z1Z2 via ancilla q4
sim.h(4);
sim.cx(1, 4);
sim.cx(2, 4);
sim.h(4);
println!("After Z1Z2 syndrome circuit (before measurement):");
println!("Stabilizers:\n{}", sim.stab_tableau());

let s1: SymbolicMeasurementResult = sim.mz(4);
println!("Syndrome S1 (Z1Z2) = {:?}, det={}", s1.outcome, s1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure data qubits in Z basis
let d0: SymbolicMeasurementResult = sim.mz(0);
let d1: SymbolicMeasurementResult = sim.mz(1);
let d2: SymbolicMeasurementResult = sim.mz(2);

println!("Data qubit measurements:");
println!("  D0 (q0) = {:?}, det={}", d0.outcome, d0.is_deterministic);
println!("  D1 (q1) = {:?}, det={}", d1.outcome, d1.is_deterministic);
println!("  D2 (q2) = {:?}, det={}", d2.outcome, d2.is_deterministic);

println!("\n=== Summary ===");
println!("Measurement indices: S0=0, S1=1, D0=2, D1=3, D2=4");
println!("\nSyndromes should be deterministic {{}} (no errors in code space).");
println!("Data qubits: D0 is random, D1 and D2 are determined by D0 and syndromes.");
println!("\nDetectors: {{S0=0, S1=0, D1^D0=0, D2^D0=0}}")

=== 3-Qubit Repetition Code: Logical |+_L⟩ ===

After encoding |+_L⟩ = (|000⟩ + |111⟩)/√2:
Stabilizers:
{} XXXII
{} ZZIII
{} ZIZII
{} IIIZI
{} IIIIZ

After Z0Z1 syndrome circuit (before measurement):
Stabilizers:
{} XXXII
{} ZZIII
{} ZIZII
{} IIIZI
{} IIIIZ

Syndrome S0 (Z0Z1) = {}, det=true
Stabilizers:
{} XXXII
{} ZZIII
{} ZIZII
{} IIIZI
{} IIIIZ

After Z1Z2 syndrome circuit (before measurement):
Stabilizers:
{} XXXII
{} ZZIII
{} ZIZII
{} IIIZI
{} IIIIZ

Syndrome S1 (Z1Z2) = {}, det=true
Stabilizers:
{} XXXII
{} ZZIII
{} ZIZII
{} IIIZI
{} IIIIZ

Data qubit measurements:
  D0 (q0) = {2}, det=false
  D1 (q1) = {2}, det=true
  D2 (q2) = {2}, det=true

=== Summary ===
Measurement indices: S0=0, S1=1, D0=2, D1=3, D2=4

Syndromes should be deterministic {} (no errors in code space).
Data qubits: D0 is random, D1 and D2 are determined by D0 and syndromes.

Detectors: {S0=0, S1=0, D1^D0=0, D2^D0=0}


()

In [11]:
// More complex example: 3-qubit circuit with interesting measurement dependencies
// 
// Circuit:
//   q0: --H--@-------M0
//            |
//   q1: -----X--H--@-M1
//                  |
//   q2: -----------X-M2
//
// This creates a chain where:
// - q0 and q1 become entangled (Bell pair)
// - Then q1 and q2 become entangled
// - Measuring in this order should show interesting dependencies

let mut sim = StdSymbolicSparseStab::new(3);

// Create the entanglement chain
sim.h(0);        // q0 in superposition
sim.cx(0, 1);    // Entangle q0-q1
sim.h(1);        // q1 in superposition (relative to q0)
sim.cx(1, 2);    // Entangle q1-q2

// Measure all qubits
let m0: SymbolicMeasurementResult = sim.mz(0);
let m1: SymbolicMeasurementResult = sim.mz(1);
let m2: SymbolicMeasurementResult = sim.mz(2);

println!("3-Qubit Chain Measurement Results:");
println!("  M0 (qubit 0): {:?} (deterministic: {})", m0.outcome, m0.is_deterministic);
println!("  M1 (qubit 1): {:?} (deterministic: {})", m1.outcome, m1.is_deterministic);
println!("  M2 (qubit 2): {:?} (deterministic: {})", m2.outcome, m2.is_deterministic);

println!("\nInterpretation:");
println!("  - M0 outcome is random and causes stab Z0 with sign: {:?}", m0.outcome);
println!("  - M1 outcome is random and causes stab Z1 with sign: {:?}", m1.outcome);
println!("  - M2 outcome is deterministic and depends on: {:?}", m2.outcome);

// Show what XOR means
if m2.outcome.len() == 2 {
    println!("\n  M2 = {{0, 1}} means: M2_outcome = M0_outcome XOR M1_outcome");
}

3-Qubit Chain Measurement Results:
  M0 (qubit 0): {0} (deterministic: false)
  M1 (qubit 1): {1} (deterministic: false)
  M2 (qubit 2): {1} (deterministic: true)

Interpretation:
  - M0 outcome is random and causes stab Z0 with sign: {0}
  - M1 outcome is random and causes stab Z1 with sign: {1}
  - M2 outcome is deterministic and depends on: {1}


()

In [10]:
// Trace through the 3-qubit circuit step by step with tableau display

let mut sim = StdSymbolicSparseStab::new(3);

println!("Initial state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(0);
println!("After H(0):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(0, 1);
println!("After CX(0,1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(1);
println!("After H(1):");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.cx(1, 2);
println!("After CX(1,2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After measuring qubit 0 (M0={:?}, det={}):", m0.outcome, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After measuring qubit 1 (M1={:?}, det={}):", m1.outcome, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After measuring qubit 2 (M2={:?}, det={}):", m2.outcome, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

Initial state:
Stabilizers:
{} ZII
{} IZI
{} IIZ

After H(0):
Stabilizers:
{} XII
{} IZI
{} IIZ

After CX(0,1):
Stabilizers:
{} XXI
{} ZZI
{} IIZ

After H(1):
Stabilizers:
{} XZI
{} ZXI
{} IIZ

After CX(1,2):
Stabilizers:
{} XZI
{} ZXX
{} IZZ

After measuring qubit 0 (M0={0}, det=false):
Stabilizers:
{0} ZII
{} ZXX
{} IZZ

After measuring qubit 1 (M1={1}, det=false):
Stabilizers:
{0} ZII
{1} IZI
{} IZZ

After measuring qubit 2 (M2={1}, det=true):
Stabilizers:
{0} ZII
{1} IZI
{} IZZ



In [8]:
// Example where M2 depends on BOTH M0 and M1 (XOR)
//
// Circuit: Create a 3-qubit GHZ-like state, then do local rotations
//
//   q0: --H--@--------M0
//            |
//   q1: -----X--@-----M1
//               |
//   q2: --------X--H--M2
//
// The key insight: after CX gates, we have correlations.
// The H on q2 at the end converts Z2 to X2, which should
// create an XOR dependency when measured in Z basis.

let mut sim = StdSymbolicSparseStab::new(3);

println!("=== Circuit where M2 = M0 XOR M1 ===\n");

sim.h(0);
sim.cx(0, 1);
sim.cx(1, 2);
println!("After H(0), CX(0,1), CX(1,2) - GHZ state:");
println!("Stabilizers:\n{}", sim.stab_tableau());

sim.h(2);
println!("After H(2):");
println!("Stabilizers:\n{}", sim.stab_tableau());

// Now measure
let m0: SymbolicMeasurementResult = sim.mz(0);
println!("After M0 (outcome={:?}, det={}):", m0.outcome, m0.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m1: SymbolicMeasurementResult = sim.mz(1);
println!("After M1 (outcome={:?}, det={}):", m1.outcome, m1.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

let m2: SymbolicMeasurementResult = sim.mz(2);
println!("After M2 (outcome={:?}, det={}):", m2.outcome, m2.is_deterministic);
println!("Stabilizers:\n{}", sim.stab_tableau());

println!("Summary:");
println!("  M0 = {:?}", m0.outcome);
println!("  M1 = {:?}", m1.outcome);
println!("  M2 = {:?}", m2.outcome);
if m2.outcome.len() == 2 && m2.outcome.contains(&0) && m2.outcome.contains(&1) {
    println!("\n  M2 = {{0,1}} means: M2_outcome = M0_outcome XOR M1_outcome");
}

{} IIZ

After H(1):
Stabilizers:
{} XZI
{} ZXI
{} IIZ

After CX(1,2):
Stabilizers:
{} XZI
{} ZXX
{} IZZ

After measuring qubit 0 (M0={0}, det=false):
Stabilizers:
{0} ZII
{} ZXX
{} IZZ

After measuring qubit 1 (M1={1}, det=false):
Stabilizers:
{0} ZII
{1} IZI
{} IZZ

After measuring qubit 2 (M2={1}, det=true):
Stabilizers:
{0} ZII
{1} IZI
{} IZZ

=== Circuit where M2 = M0 XOR M1 ===

After H(0), CX(0,1), CX(1,2) - GHZ state:
Stabilizers:
{} XXX
{} ZZI
{} IZZ

After H(2):
Stabilizers:
{} XXZ
{} ZZI
{} IZX

After M0 (outcome={0}, det=false):
Stabilizers:
{0} ZII
{} ZZI
{} IZX

After M1 (outcome={0}, det=true):
Stabilizers:
{0} ZII
{} ZZI
{} IZX

After M2 (outcome={2}, det=false):
Stabilizers:
{0} ZII
{} ZZI
{2} IIZ

Summary:
  M0 = {0}
  M1 = {0}
  M2 = {2}


()